# Практика · Instance segmentation: маска на кожен предмет

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> 🔌 **Мережа не потрібна.** Усі сцени зошит малює формулами. Досить `torch`,
> `torchvision`, `numpy`, `opencv-python` і `matplotlib`.

> ⏱ Зошит навчає **девʼять мереж**: шість детекторів із гілкою маски й три мережі
> голосування, по три зерна на кожну настройку. Заміряно чотири рази: **сам зошит
> друкує в кінці 283-326 секунд**, а перевірка разом із запуском ядра — **від 324
> до 418**. Тобто пʼять-сім хвилин на чотирьох ядрах без відеокарти, в один потік,
> залежно від того, чим ще зайнята машина. Дві клітинки з навчанням дають більшу
> частину цього часу; решта зошита йде за секунди.

Семантична маска з [теми 29](../29-semantic-segmentation/lecture.html) каже, **якого
класу** піксель. Ця тема питає інакше: **якому предмету** він належить. Що зробимо:

1. зберемо сцени з масками **кожного предмета окремо** й порахуємо головне число
   теми — **скільки предметів злипається**;
2. поміряємо наївну базу: **`cv2.connectedComponents`** на семантичній масці;
3. напишемо **свої mask IoU і mask AP**, звіримо їх із прикладами, порахованими
   руками, і побачимо, чим mask AP відрізняється від box AP;
4. додамо до anchor-free детектора з [теми 26](../26-one-stage/lecture.html)
   **гілку маски** й навчимо його, три зерна;
5. порахуємо **стелю роздільності маски 28×28** без жодного навчання;
6. порівняємо **RoI-Align і RoI-пулінг** саме на масках;
7. розберемо `maskrcnn_resnet50_fpn` по частинах;
8. зробимо підхід **знизу вгору** — голосування пікселів за центр — і порівняємо
   всі три способи.

In [ ]:
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import matplotlib.pyplot as plt
from torchvision.ops import box_iou, nms, roi_align, roi_pool

# один потік дає і швидкість на маленьких тензорах, і повторюваність чисел:
# під кількома потоками float-суми йдуть в іншому порядку
torch.set_num_threads(1)

# скільки й як довго вчимо — зібрано в одному місці, щоб було легко переміряти
DET_EPOCHS, DET_BATCH = 10, 16                # детектор із гілкою маски
VOTE_EPOCHS, VOTE_BATCH = 8, 8                # мережа голосування

NOTEBOOK_STARTED = time.time()
print("torch   ", torch.__version__)
print("cv2     ", cv2.__version__)
print("numpy   ", np.__version__)
print("потоків ", torch.get_num_threads())

## 1 · Сцени з маскою кожного предмета окремо

Генератор — **канонічний, із [теми 28](../28-detection-practice/lecture.html)**, і
переписувати його не можна: щойно змінити правило розміщення фігур, попливуть
частки класів і всі числа блоку разом із ними.

Що додано, і тільки це: разом із рамкою й класом ми зберігаємо **маску кожної
фігури окремо**. Пізніша фігура затуляє ранню, тож маска предмета — це його
**видима** частина, а рамка рахується з тієї самої видимої маски.

Параметр `max_objects` каже, скільки фігур кидати на полотно. Значення 3 —
канонічне; значення 6 дає **щільні** сцени, заради яких і затіяна тема.

In [ ]:
SIZE = 64                                   # сторона полотна в пікселях
CLASS_NAMES = ["коло", "квадрат", "трикутник"]
CLASS_COUNT = 3
GRID = 8                                    # карта ознак 8×8
STRIDE = SIZE / GRID                        # крок карти: 8 пікселів на клітинку


def shape_mask(kind, center_x, center_y, radius):
    '''Маска однієї фігури на полотні 64×64 — та сама геометрія, що в блоці 5.'''
    ys, xs = np.mgrid[0:SIZE, 0:SIZE]
    if kind == 0:                                    # коло
        return (xs - center_x) ** 2 + (ys - center_y) ** 2 <= radius * radius
    if kind == 1:                                    # квадрат
        return (np.abs(xs - center_x) <= radius) & (np.abs(ys - center_y) <= radius)
    # трикутник: ширина росте згори вниз
    return ((ys - center_y + radius >= 0) & (ys - center_y <= radius)
            & (np.abs(xs - center_x) <= (ys - center_y + radius) / 2.0))


def box_from_mask(mask):
    '''Рамка з маски: край + 1 по правому й нижньому боці, як в угоді COCO.'''
    ys, xs = np.nonzero(mask)
    return [float(xs.min()), float(ys.min()), float(xs.max() + 1), float(ys.max() + 1)]


def make_scene(rng, max_objects=3, overlap_limit=0.45):
    '''Канонічна сцена теми 28 плюс маска кожної фігури окремо.'''
    image = np.zeros((SIZE, SIZE), np.float32)
    boxes, labels, masks = [], [], []
    for _slot in range(int(rng.integers(1, max_objects + 1))):
        for _attempt in range(40):
            radius = int(rng.integers(6, 11))
            center_x = int(rng.integers(radius + 1, SIZE - radius - 1))
            center_y = int(rng.integers(radius + 1, SIZE - radius - 1))
            kind = int(rng.integers(0, 3))
            mask = shape_mask(kind, center_x, center_y, radius)
            box = box_from_mask(mask)

            # канонічне правило блоку: нова фігура не перекриває стару
            # більше ніж на 45 % своєї площі
            too_close = False
            for previous in boxes:
                width = max(0, min(box[2], previous[2]) - max(box[0], previous[0]))
                height = max(0, min(box[3], previous[3]) - max(box[1], previous[1]))
                if width * height > overlap_limit * (box[2] - box[0]) * (box[3] - box[1]):
                    too_close = True
                    break
            if too_close:
                continue

            image[mask] = 1.0
            boxes.append(box)
            labels.append(kind)
            masks.append(mask)
            break
    image = np.clip(image + rng.normal(0, 0.12, (SIZE, SIZE)).astype(np.float32), 0, 1)
    return (image, np.array(boxes, np.float32).reshape(-1, 4),
            np.array(labels, np.int64), masks)


def visible_masks(masks):
    '''Пізніші фігури затуляють ранніх — лишаємо тільки видимі пікселі.'''
    out = []
    for index, mask in enumerate(masks):
        visible = mask.copy()
        for later in masks[index + 1:]:
            visible &= ~later
        out.append(visible)
    return out


def make_dataset(seed, count, max_objects=3):
    '''Набір сцен: картинка, видимі маски, рамки з них, семантична маска.'''
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(count):
        image, _boxes, labels, masks = make_scene(rng, max_objects)
        visible = visible_masks(masks)
        # предмет, затулений цілком, із розмітки зникає — його просто не видно
        keep = [i for i in range(len(visible)) if visible[i].sum() > 0]
        visible = [visible[i] for i in keep]
        labels = labels[keep]
        boxes = np.array([box_from_mask(m) for m in visible], np.float32).reshape(-1, 4)
        segmentation = np.zeros((SIZE, SIZE), np.int64)
        for index, mask in enumerate(visible):
            segmentation[mask] = labels[index] + 1
        out.append(dict(image=image, masks=visible, labels=labels,
                        boxes=boxes, seg=segmentation))
    return out


started = time.time()
sparse_train = make_dataset(42, 240)                    # канонічні, до 3 предметів
sparse_test = make_dataset(7, 120)
dense_train = make_dataset(42, 240, max_objects=6)      # щільні, до 6 предметів
dense_test = make_dataset(7, 120, max_objects=6)

print("згенеровано за %.2f с" % (time.time() - started))
for name, data in (("рідкі навчальні", sparse_train), ("рідкі перевірні", sparse_test),
                   ("щільні навчальні", dense_train), ("щільні перевірні", dense_test)):
    total = sum(len(scene["labels"]) for scene in data)
    print("  %-17s сцен %3d, предметів %3d, у середньому %.2f на сцену"
          % (name, len(data), total, total / len(data)))

### Звірка з темою 28

Найнадійніша перевірка того, що генератор той самий, — прогнати його поруч зі
старим і порівняти побітово. Картинка й мітки мусять збігтися **точно**; рамки
можуть відрізнятись лише там, де одна фігура затулила іншу, бо тепер вони
рахуються з видимої маски, а не з повної фігури.

In [ ]:
def make_scene_topic28(rng):
    '''Копія генератора теми 28 — щоб було з чим звіряти.'''
    image = np.zeros((SIZE, SIZE), np.float32)
    boxes, labels = [], []
    for _slot in range(int(rng.integers(1, 4))):
        for _attempt in range(40):
            radius = int(rng.integers(6, 11))
            center_x = int(rng.integers(radius + 1, SIZE - radius - 1))
            center_y = int(rng.integers(radius + 1, SIZE - radius - 1))
            kind = int(rng.integers(0, 3))
            mask = shape_mask(kind, center_x, center_y, radius)
            box = box_from_mask(mask)
            too_close = False
            for previous in boxes:
                width = max(0, min(box[2], previous[2]) - max(box[0], previous[0]))
                height = max(0, min(box[3], previous[3]) - max(box[1], previous[1]))
                if width * height > 0.45 * (box[2] - box[0]) * (box[3] - box[1]):
                    too_close = True
                    break
            if too_close:
                continue
            image[mask] = 1.0
            boxes.append(box)
            labels.append(kind)
            break
    image = np.clip(image + rng.normal(0, 0.12, (SIZE, SIZE)).astype(np.float32), 0, 1)
    return image, np.array(boxes, np.float32).reshape(-1, 4), np.array(labels, np.int64)


old_rng = np.random.default_rng(42)
new_rng = np.random.default_rng(42)
changed_boxes, hidden = 0, 0
for _ in range(240):
    old_image, old_boxes, old_labels = make_scene_topic28(old_rng)
    new_image, _new_boxes, new_labels, new_masks = make_scene(new_rng)
    assert np.array_equal(old_image, new_image), "картинка розійшлася з темою 28!"
    assert np.array_equal(old_labels, new_labels), "мітки розійшлися з темою 28!"
    for index, mask in enumerate(visible_masks(new_masks)):
        if mask.sum() == 0:
            hidden += 1
        elif not np.allclose(old_boxes[index], box_from_mask(mask)):
            changed_boxes += 1

print("✅ 240 сцен збігаються з темою 28 побітово: картинка і мітки")
print("рамок, що змінились через затуляння: %d" % changed_boxes)
print("предметів, затулених цілком і викинутих із розмітки: %d" % hidden)

### Як це виглядає

Зліва — картинка, справа — та сама сцена, розібрана на предмети: кожен предмет
своїм кольором. Верхній ряд — рідкі сцени, нижній — щільні.

In [ ]:
figure, axes = plt.subplots(2, 6, figsize=(15, 5.4))
for column in range(3):
    for row, data in enumerate((sparse_test, dense_test)):
        scene = data[column]
        instance_map = np.zeros((SIZE, SIZE), np.int64)
        for index, mask in enumerate(scene["masks"]):
            instance_map[mask] = index + 1
        axes[row, 2 * column].imshow(scene["image"], cmap="gray", vmin=0, vmax=1)
        axes[row, 2 * column].set_title("сцена %d" % column, fontsize=9)
        axes[row, 2 * column + 1].imshow(instance_map, cmap="tab10", vmin=0, vmax=9)
        axes[row, 2 * column + 1].set_title("%d предмет(ів)" % len(scene["masks"]),
                                            fontsize=9)
        axes[row, 2 * column].axis("off")
        axes[row, 2 * column + 1].axis("off")
plt.tight_layout()
plt.show()
print("верхній ряд — рідкі сцени, нижній — щільні")

## 2 · Головне число теми: скільки предметів злипається

Два питання до кожного набору сцен.

**Перше.** Яка частка предметів **торкається** іншого предмета? Торкаються — це
або перекриваються, або стоять поруч так, що між ними немає жодного пікселя фону
(8-звʼязність: сусідами вважаються й діагональні пікселі).

**Друге.** Скільки **звʼязних плям** дає семантична маска? Пляму рахуємо окремо в
кожному класі — інакше коло, що торкається квадрата, злилося б із ним.

In [ ]:
CROSS = np.ones((3, 3), np.uint8)          # околиця 8-звʼязності


def touches(first, second):
    '''Дві маски торкаються, якщо перетинаються або стоять упритул.'''
    grown = cv2.dilate(first.astype(np.uint8), CROSS).astype(bool)
    return bool((grown & second).any())


def count_blobs(segmentation):
    '''Скільки звʼязних плям у семантичній масці — по кожному класу окремо.'''
    total = 0
    for class_index in range(1, CLASS_COUNT + 1):
        found, _marks = cv2.connectedComponents(
            (segmentation == class_index).astype(np.uint8), connectivity=8)
        total += found - 1                  # мінус фон
    return total


def scene_report(name, data):
    objects = same_class = any_class = blobs = exact = 0
    for scene in data:
        masks, labels = scene["masks"], scene["labels"]
        objects += len(masks)
        for i in range(len(masks)):
            same = any(touches(masks[i], masks[j]) for j in range(len(masks))
                       if j != i and labels[j] == labels[i])
            anyone = any(touches(masks[i], masks[j]) for j in range(len(masks)) if j != i)
            same_class += int(same)
            any_class += int(anyone)
        found = count_blobs(scene["seg"])
        blobs += found
        exact += int(found == len(masks))
    print("%s: %d сцен, %d предметів" % (name, len(data), objects))
    print("   торкається предмета свого класу : %3d (%.1f %%)"
          % (same_class, 100 * same_class / objects))
    print("   торкається будь-якого предмета  : %3d (%.1f %%)"
          % (any_class, 100 * any_class / objects))
    print("   звʼязних плям у семантичній масці: %3d проти %d істинних"
          % (blobs, objects))
    print("   сцен, де кількість плям збіглася : %3d із %d (%.1f %%)"
          % (exact, len(data), 100 * exact / len(data)))
    return dict(objects=objects, same=same_class, blobs=blobs)


sparse_stats = scene_report("рідкі перевірні сцени (до 3 предметів)", sparse_test)
print()
dense_stats = scene_report("щільні перевірні сцени (до 6 предметів)", dense_test)
print()
print("Кожна пара дотичних предметів одного класу коштує приблизно однієї плями:")
print("   рідкі  — торкається %d, тобто близько %d пар; плям бракує %d"
      % (sparse_stats["same"], sparse_stats["same"] // 2,
         sparse_stats["objects"] - sparse_stats["blobs"]))
print("   щільні — торкається %d, тобто близько %d пар; плям бракує %d"
      % (dense_stats["same"], dense_stats["same"] // 2,
         dense_stats["objects"] - dense_stats["blobs"]))
print("Точної рівності немає у двох боки: ланцюжок із трьох дотичних фігур")
print("коштує двох плям, а фігура, розрізана навпіл чужою, додає зайву.")

## 3 · Наївна база: звʼязні компоненти

Розберімо семантичну маску на плями й оголосімо кожну пляму предметом. Це чесна
базова лінія: один рядок коду, жодного навчання. Далі все, що ми побудуємо, має
вигравати саме в неї.

In [ ]:
def connected_predictor(segmentation):
    '''Кожна звʼязна пляма семантичної маски стає окремим предметом.'''
    masks, labels, boxes, scores = [], [], [], []
    for class_index in range(1, CLASS_COUNT + 1):
        found, marks = cv2.connectedComponents(
            (segmentation == class_index).astype(np.uint8), connectivity=8)
        for blob in range(1, found):
            mask = marks == blob
            masks.append(mask)
            labels.append(class_index - 1)
            boxes.append(box_from_mask(mask))
            # упевненості тут узятись нізвідки, тож беремо площу плями:
            # більша пляма — надійніший кандидат
            scores.append(float(mask.sum()))
    return dict(masks=masks, labels=np.array(labels, np.int64),
                boxes=np.array(boxes, np.float32).reshape(-1, 4),
                scores=np.array(scores, np.float32))


def truth_of(scene):
    '''Істина в тому самому форматі, що й передбачення.'''
    return dict(masks=scene["masks"], labels=scene["labels"], boxes=scene["boxes"],
                scores=np.ones(len(scene["labels"]), np.float32))


sparse_truth = [truth_of(scene) for scene in sparse_test]
dense_truth = [truth_of(scene) for scene in dense_test]
sparse_blobs = [connected_predictor(scene["seg"]) for scene in sparse_test]
dense_blobs = [connected_predictor(scene["seg"]) for scene in dense_test]
print("плям видано: рідкі %d, щільні %d"
      % (sum(len(p["masks"]) for p in sparse_blobs),
         sum(len(p["masks"]) for p in dense_blobs)))

### Скільки предметів вони відновлюють

Питання ставимо так: для скількох істинних предметів знайшлася пляма, що
збігається з ним хоча б наполовину. «Збігається» — це `IoU` двох масок, і саме
його ми зараз і напишемо.

In [ ]:
def mask_iou(first, second):
    '''IoU двох масок: спільних пікселів поділити на всі зайняті хоч однією.'''
    intersection = np.logical_and(first, second).sum()
    union = np.logical_or(first, second).sum()
    return float(intersection) / float(union) if union else 0.0


# звірка на прикладі, порахованому руками:
# коло радіуса 5 займає 81 піксель, його рамка — квадрат 11×11 = 121 піксель,
# коло цілком усередині рамки, тож IoU = 81 / 121
hand_circle = shape_mask(0, 20, 20, 5)
hand_box = np.zeros((SIZE, SIZE), bool)
hand_box[15:26, 15:26] = True

print("пікселів у колі радіуса 5 : %d  (рахували руками: 81)" % hand_circle.sum())
print("пікселів у його рамці     : %d  (рахували руками: 121)" % hand_box.sum())
print("mask IoU                  : %.4f  (рахували руками: %.4f)"
      % (mask_iou(hand_circle, hand_box), 81 / 121))
assert hand_circle.sum() == 81 and hand_box.sum() == 121, "геометрія прикладу зламалась!"
assert np.isclose(mask_iou(hand_circle, hand_box), 81 / 121), "наше IoU розійшлося!"
print("✅ збігається")

In [ ]:
def recovery_report(name, data, predictions):
    total = touching = touching_ok = free = free_ok = 0
    hits = {0.50: 0, 0.75: 0, 0.90: 0}
    for scene, prediction in zip(data, predictions):
        masks, labels = scene["masks"], scene["labels"]
        best = np.zeros(len(masks))
        for j, truth_mask in enumerate(masks):
            for predicted_mask in prediction["masks"]:
                best[j] = max(best[j], mask_iou(predicted_mask, truth_mask))
        for j in range(len(masks)):
            total += 1
            for threshold in hits:
                hits[threshold] += int(best[j] >= threshold)
            same = any(touches(masks[j], masks[k]) for k in range(len(masks))
                       if k != j and labels[k] == labels[j])
            if same:
                touching += 1
                touching_ok += int(best[j] >= 0.5)
            else:
                free += 1
                free_ok += int(best[j] >= 0.5)
    print("%s (%d предметів):" % (name, total))
    for threshold in (0.50, 0.75, 0.90):
        print("   відновлено при IoU не менше %.2f : %.3f" % (threshold, hits[threshold] / total))
    print("   з тих, що ні з ким не торкаються (%3d): %.3f" % (free, free_ok / free))
    print("   з тих, що торкаються свого класу (%3d): %.3f"
          % (touching, touching_ok / max(touching, 1)))


recovery_report("рідкі сцени", sparse_test, sparse_blobs)
print()
recovery_report("щільні сцени", dense_test, dense_blobs)
print()
print("Предмет, який ні з ким не торкається, звʼязні компоненти відновлюють ідеально.")
print("Той, що торкається сусіда свого класу, — у третині-половині випадків.")

## 4 · Своя mask AP

`mask AP` — це `box AP` із [теми 23](../23-detection-setup/lecture.html), у якій
IoU рахується по масках. Механіка та сама, тож напишемо її один раз і
перемикатимемо лише спосіб рахувати збіг.

Одна технічна деталь заради швидкості: IoU кожної пари «прогноз-істина» рахуємо
**один раз** і складаємо в кошик. Тоді перебір десяти порогів для усереднення
`AP[.50:.95]` не коштує десяти перерахунків.

In [ ]:
THRESHOLDS = np.arange(0.50, 0.96, 0.05)      # ті самі десять порогів, що в COCO


def mask_iou_matrix(predicted_masks, true_masks):
    out = np.zeros((len(predicted_masks), len(true_masks)))
    for i, predicted in enumerate(predicted_masks):
        for j, truth in enumerate(true_masks):
            out[i, j] = mask_iou(predicted, truth)
    return out


def box_iou_matrix(predicted_boxes, true_boxes):
    if len(predicted_boxes) == 0 or len(true_boxes) == 0:
        return np.zeros((len(predicted_boxes), len(true_boxes)))
    return box_iou(torch.as_tensor(np.asarray(predicted_boxes), dtype=torch.float32),
                   torch.as_tensor(np.asarray(true_boxes), dtype=torch.float32)).numpy()


def collect_overlaps(predictions, truths, kind):
    '''IoU кожної пари рахуємо один раз і кладемо в кошик свого класу.'''
    baskets = []
    for class_index in range(CLASS_COUNT):
        scenes, truth_count = [], 0
        for prediction, truth in zip(predictions, truths):
            chosen = [i for i, c in enumerate(prediction["labels"]) if c == class_index]
            keep = [j for j, c in enumerate(truth["labels"]) if c == class_index]
            truth_count += len(keep)
            if not chosen:
                continue
            if kind == "box":
                overlaps = box_iou_matrix([prediction["boxes"][i] for i in chosen],
                                          [truth["boxes"][j] for j in keep])
            else:
                overlaps = mask_iou_matrix([prediction["masks"][i] for i in chosen],
                                           [truth["masks"][j] for j in keep])
            scores = np.asarray([prediction["scores"][i] for i in chosen], np.float64)
            order = np.argsort(-scores, kind="stable")      # найвпевненіші попереду
            scenes.append((scores[order], overlaps[order]))
        baskets.append((scenes, truth_count))
    return baskets


def greedy_hits(overlaps, threshold):
    '''Кожен прогноз по черзі забирає найкращу ще вільну істину — правило теми 23.'''
    hits = np.zeros(len(overlaps))
    if len(overlaps) == 0 or overlaps.shape[1] == 0:
        return hits
    taken = np.zeros(overlaps.shape[1], bool)
    for position in range(len(overlaps)):
        row = overlaps[position].copy()
        row[taken] = -1.0                    # зайняті істини більше не пропонуємо
        best = int(row.argmax())
        if row[best] >= threshold:
            taken[best] = True
            hits[position] = 1.0
    return hits


def average_precision(scores, hits, truth_count):
    '''AP як площа під огинальною кривої точність-повнота.'''
    if truth_count == 0:
        return float("nan")
    if len(scores) == 0:
        return 0.0
    order = np.argsort(-np.asarray(scores), kind="stable")
    ordered = np.asarray(hits)[order]
    running_hits = np.cumsum(ordered)
    running_misses = np.cumsum(1 - ordered)
    precision = running_hits / np.maximum(running_hits + running_misses, 1e-9)
    recall = running_hits / truth_count
    envelope = np.maximum.accumulate(precision[::-1])[::-1]
    steps = np.diff(np.concatenate([[0.0], recall]))
    return float(np.sum(steps * envelope))


def ap_at(baskets, threshold):
    '''AP при одному порозі, усереднена по трьох класах.'''
    values = []
    for scenes, truth_count in baskets:
        scores, hits = [], []
        for scene_scores, overlaps in scenes:
            hits.extend(greedy_hits(overlaps, threshold))
            scores.extend(scene_scores)
        values.append(average_precision(scores, hits, truth_count))
    values = [v for v in values if not np.isnan(v)]
    return float(np.mean(values)) if values else 0.0


def ap_mean(baskets):
    '''COCO-подібне усереднення по порогах 0.50…0.95 із кроком 0.05.'''
    return float(np.mean([ap_at(baskets, threshold) for threshold in THRESHOLDS]))


print("функції готові")

### Звірка AP на прикладі, порахованому руками

Три прогнози з упевненостями 0.9, 0.8 і 0.3 на дві істини. Перший влучає в першу
істину, другий цілиться в неї ж (але вона вже зайнята — це помилка), третій
влучає в другу істину.

- влучання за спаданням упевненості: **1, 0, 1**;
- точність після кожного кроку: 1/1 = 1, 1/2 = 0.5, 2/3 = 0.667;
- повнота: 0.5, 0.5, 1.0;
- огинальна (максимум точності праворуч): 1, 0.667, 0.667;
- приріст повноти на кожному кроці: 0.5, 0, 0.5;
- **AP = 0.5·1 + 0·0.667 + 0.5·0.667 = 0.8333**.

In [ ]:
hand_scores = np.array([0.9, 0.8, 0.3])
hand_hits = np.array([1.0, 0.0, 1.0])
hand_ap = average_precision(hand_scores, hand_hits, truth_count=2)

print("AP на прикладі: %.4f  (рахували руками: 0.8333)" % hand_ap)
assert np.isclose(hand_ap, 0.5 + 0.5 * 2 / 3), "наша AP розійшлася з ручним підрахунком!"
print("✅ збігається")
print()

# і перевірка самої реалізації: ідеальний прогноз мусить дати рівно одиницю
perfect = collect_overlaps(dense_truth, dense_truth, "mask")
print("ідеальні маски проти себе: mask AP@0.5 = %.4f, AP[.50:.95] = %.4f"
      % (ap_at(perfect, 0.5), ap_mean(perfect)))
assert np.isclose(ap_at(perfect, 0.5), 1.0), "ідеальний прогноз мусить дати одиницю!"
print("✅ реалізація не бреше сама собі")

### Замір: box AP і mask AP на тих самих передбаченнях

Найважливіший рядок — **«маска = вся рамка»**: рамки ідеальні до пікселя, а
маскою оголошено весь прямокутник рамки. Так поводився б детектор без гілки
маски, якби його змусили видати маску.

In [ ]:
def filled_box_prediction(scene):
    '''Рамки ідеальні, маска — увесь прямокутник рамки.'''
    masks = []
    for box in scene["boxes"]:
        mask = np.zeros((SIZE, SIZE), bool)
        mask[int(box[1]):int(np.ceil(box[3])), int(box[0]):int(np.ceil(box[2]))] = True
        masks.append(mask)
    return dict(masks=masks, labels=scene["labels"], boxes=scene["boxes"],
                scores=np.ones(len(scene["labels"]), np.float32))


def shifted_prediction(scene, pixels):
    '''Форма правильна, але зсунута на кілька пікселів — рамка їде разом із нею.'''
    masks, boxes = [], []
    for mask in scene["masks"]:
        moved = np.zeros((SIZE, SIZE), bool)
        moved[pixels:, pixels:] = mask[:SIZE - pixels, :SIZE - pixels]
        if moved.sum() == 0:
            moved = mask.copy()
        masks.append(moved)
        boxes.append(box_from_mask(moved))
    return dict(masks=masks, labels=scene["labels"],
                boxes=np.array(boxes, np.float32).reshape(-1, 4),
                scores=np.ones(len(scene["labels"]), np.float32))


print("наскільки прямокутник рамки схожий на саму фігуру:")
by_class = {0: [], 1: [], 2: []}
for scene in dense_test:
    prediction = filled_box_prediction(scene)
    for j in range(len(scene["masks"])):
        by_class[int(scene["labels"][j])].append(
            mask_iou(prediction["masks"][j], scene["masks"][j]))
for class_index in range(CLASS_COUNT):
    print("   %-11s IoU %.4f" % (CLASS_NAMES[class_index],
                                 float(np.mean(by_class[class_index]))))
print()

variants = [("ідеальні маски", dense_truth),
            ("маска = вся рамка", [filled_box_prediction(s) for s in dense_test]),
            ("зсув форми на 1 px", [shifted_prediction(s, 1) for s in dense_test]),
            ("зсув форми на 2 px", [shifted_prediction(s, 2) for s in dense_test]),
            ("зсув форми на 3 px", [shifted_prediction(s, 3) for s in dense_test]),
            ("звʼязні плями", dense_blobs)]

print("  передбачення         |  box@.5 box@.75  box@av | msk@.5 msk@.75  msk@av")
print("  " + "-" * 70)
for name, predictions in variants:
    boxes_basket = collect_overlaps(predictions, dense_truth, "box")
    masks_basket = collect_overlaps(predictions, dense_truth, "mask")
    print("  %-20s | %.4f %.4f  %.4f | %.4f  %.4f  %.4f"
          % (name, ap_at(boxes_basket, 0.5), ap_at(boxes_basket, 0.75), ap_mean(boxes_basket),
             ap_at(masks_basket, 0.5), ap_at(masks_basket, 0.75), ap_mean(masks_basket)))
print()
print("Рамки в другому рядку ідеальні — box AP дорівнює одиниці при будь-якому порозі.")
print("mask AP при цьому падає майже вдвічі: рамка не бачить форми.")
print("А в останньому рядку навпаки — mask AP вища. Метрика сувора до тієї помилки,")
print("яку саме робить модель, і однієї «суворішої завжди» метрики не існує.")

## 5 · Зверху вниз: детектор із гілкою маски

Беремо anchor-free детектор із [теми 26](../26-one-stage/lecture.html) **без
жодної зміни**: карта 8×8, три числа класу в кожній позиції, чотири відстані до
країв рамки й одне число centerness. Тіло тепер віддає ще й проміжну карту 16×16 —
з неї гілка маски вирізає ділянку під рамкою.

Гілка маски — Mask R-CNN у мініатюрі: `RoI-Align` 14×14 → дві згортки →
`ConvTranspose2d` подвоює до 28×28 → `Conv 1×1` дає по каналу на клас.

In [ ]:
POINTS = torch.tensor([[(col + 0.5) * STRIDE, (row + 0.5) * STRIDE]
                       for row in range(GRID) for col in range(GRID)],
                      dtype=torch.float32)
PRIOR = 0.01                                   # бажана ймовірність предмета на старті
BIAS_INIT = -math.log((1 - PRIOR) / PRIOR)
MASK_SIDE = 28                                 # той самий розмір, що в Mask R-CNN
FEATURE_SCALE = 0.25                           # карта 16×16 відносно входу 64×64


def focal_loss(logits, targets, alpha=0.25, gamma=2.0):
    '''Стійка версія: крос-ентропію рахуємо з логітів, не переходячи через p.'''
    probability = torch.sigmoid(logits)
    cross_entropy = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p_t = probability * targets + (1 - probability) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    return alpha_t * cross_entropy * (1 - p_t) ** gamma


def free_targets(boxes, labels):
    '''Ціль anchor-free: позиція позитивна, якщо вона всередині рамки.'''
    total = POINTS.shape[0]
    positive = torch.zeros(total, dtype=torch.bool)
    class_id = torch.full((total,), -1, dtype=torch.long)
    distances = torch.zeros(total, 4)
    if len(boxes) == 0:
        return positive, class_id, distances
    truth = torch.as_tensor(np.ascontiguousarray(boxes), dtype=torch.float32)
    areas = (truth[:, 2] - truth[:, 0]) * (truth[:, 3] - truth[:, 1])
    point_x, point_y = POINTS[:, 0:1], POINTS[:, 1:2]
    left, top = point_x - truth[:, 0], point_y - truth[:, 1]
    right, bottom = truth[:, 2] - point_x, truth[:, 3] - point_y
    inside = (left > 0) & (top > 0) & (right > 0) & (bottom > 0)
    # серед рамок, що містять точку, беремо найменшу за площею
    area_or_infinity = torch.where(inside, areas.expand_as(inside),
                                   torch.full_like(inside, float("inf"),
                                                   dtype=torch.float32))
    smallest, chosen = area_or_infinity.min(dim=1)
    positive = torch.isfinite(smallest)
    index = torch.arange(total)
    stacked = torch.stack([left[index, chosen], top[index, chosen],
                           right[index, chosen], bottom[index, chosen]], dim=1)
    distances[positive] = stacked[positive] / STRIDE
    class_id[positive] = torch.as_tensor(labels, dtype=torch.long)[chosen[positive]]
    return positive, class_id, distances


def centerness_target(distances):
    '''Наскільки позиція близька до центра рамки: 1 у центрі, 0 на краю.'''
    left, top, right, bottom = distances.unbind(dim=1)
    horizontal = torch.min(left, right) / torch.max(left, right).clamp(min=1e-6)
    vertical = torch.min(top, bottom) / torch.max(top, bottom).clamp(min=1e-6)
    return torch.sqrt((horizontal * vertical).clamp(min=0))


class Body(nn.Module):
    '''Тіло теми 26; додатково віддає карту 16×16 для гілки маски.'''

    def __init__(self):
        super().__init__()

        def block(in_channels, out_channels):
            return nn.Sequential(nn.Conv2d(in_channels, out_channels, 3, padding=1),
                                 nn.BatchNorm2d(out_channels), nn.ReLU(), nn.MaxPool2d(2))

        self.stage1 = block(1, 16)
        self.stage2 = block(16, 32)
        self.stage3 = block(32, 64)
        self.tail = nn.Sequential(nn.Conv2d(64, 64, 3, padding=1),
                                  nn.BatchNorm2d(64), nn.ReLU())

    def forward(self, x):
        middle = self.stage2(self.stage1(x))            # 16×16, 32 канали
        return self.tail(self.stage3(middle)), middle


class MaskHead(nn.Module):
    '''Гілка маски: ділянка 14×14 → маска 28×28, по каналу на клас.'''

    def __init__(self, channels=32, per_class=True):
        super().__init__()
        outputs = CLASS_COUNT if per_class else 1
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(channels, channels, 2, stride=2), nn.ReLU(),
            nn.Conv2d(channels, outputs, 1))

    def forward(self, patches):
        return self.net(patches)


class MaskDetector(nn.Module):
    def __init__(self, per_class=True):
        super().__init__()
        self.body = Body()
        self.classifier = nn.Conv2d(64, CLASS_COUNT, 3, padding=1)
        self.regressor = nn.Conv2d(64, 4, 3, padding=1)
        self.centerness = nn.Conv2d(64, 1, 3, padding=1)
        nn.init.constant_(self.classifier.bias, BIAS_INIT)
        self.mask_head = MaskHead(32, per_class)
        self.per_class = per_class

    def forward(self, x):
        features, middle = self.body(x)
        count = x.shape[0]
        logits = self.classifier(features).permute(0, 2, 3, 1).reshape(count, -1,
                                                                      CLASS_COUNT)
        # відстані невідʼємні за змістом, тому пропускаємо їх крізь ReLU
        distances = F.relu(self.regressor(features)).permute(0, 2, 3, 1).reshape(count, -1, 4)
        center = self.centerness(features).permute(0, 2, 3, 1).reshape(count, -1)
        return logits, distances, center, middle

    def masks_for(self, middle, rois):
        patches = roi_align(middle, rois, output_size=(14, 14),
                            spatial_scale=FEATURE_SCALE, sampling_ratio=2, aligned=True)
        return self.mask_head(patches)


probe = MaskDetector()
body_parameters = sum(p.numel() for p in probe.body.parameters())
head_parameters = (sum(p.numel() for p in probe.classifier.parameters())
                   + sum(p.numel() for p in probe.regressor.parameters())
                   + sum(p.numel() for p in probe.centerness.parameters()))
mask_parameters = sum(p.numel() for p in probe.mask_head.parameters())
single_mask_parameters = sum(p.numel() for p in MaskDetector(False).mask_head.parameters())

print("параметрів у тілі                : %6d" % body_parameters)
print("у голові рамок                   : %6d" % head_parameters)
print("у голові масок, канал на клас    : %6d" % mask_parameters)
print("у голові масок, один канал       : %6d" % single_mask_parameters)
print("окремі канали коштують           : %6d ваг" % (mask_parameters - single_mask_parameters))
print("усього                           : %6d" % sum(p.numel() for p in probe.parameters()))

### Цілі для гілки маски

Для кожного істинного предмета вирізаємо його маску за його ж рамкою й стискаємо
до 28×28. Це і є те, чого гілка вчиться. Під час навчання рамки беремо істинні —
так само, як у статті; на перевірці подамо ті, що видасть сам детектор.

In [ ]:
def mask_targets(masks, boxes, side=MASK_SIDE):
    '''Істинна маска, вирізана за своєю рамкою й стиснута до side×side.'''
    out = []
    for mask, box in zip(masks, boxes):
        x0, y0 = int(box[0]), int(box[1])
        x1, y1 = int(np.ceil(box[2])), int(np.ceil(box[3]))
        crop = torch.from_numpy(mask[y0:y1, x0:x1].astype(np.float32))[None, None]
        out.append(F.interpolate(crop, size=(side, side), mode="bilinear",
                                 align_corners=False)[0, 0])
    if not out:
        return torch.zeros(0, side, side)
    return torch.stack(out)


def pack(data):
    '''Готуємо цілі один раз, щоб не рахувати їх у кожній епосі.'''
    images = torch.from_numpy(np.stack([scene["image"] for scene in data])[:, None])
    total = POINTS.shape[0]
    positive = torch.zeros(len(data), total, dtype=torch.bool)
    class_target = torch.zeros(len(data), total, CLASS_COUNT)
    distance_target = torch.zeros(len(data), total, 4)
    center_target = torch.zeros(len(data), total)
    gt_boxes, gt_masks, gt_labels = [], [], []
    for index, scene in enumerate(data):
        is_positive, class_id, distances = free_targets(scene["boxes"], scene["labels"])
        positive[index] = is_positive
        if is_positive.any():
            class_target[index, is_positive, class_id[is_positive]] = 1.0
            distance_target[index, is_positive] = distances[is_positive]
            center_target[index, is_positive] = centerness_target(distances[is_positive])
        gt_boxes.append(torch.as_tensor(scene["boxes"], dtype=torch.float32))
        gt_labels.append(torch.as_tensor(scene["labels"], dtype=torch.long))
        gt_masks.append(mask_targets(scene["masks"], scene["boxes"]))
    return dict(images=images, positive=positive, cls=class_target, dist=distance_target,
                center=center_target, gt_boxes=gt_boxes, gt_masks=gt_masks,
                gt_labels=gt_labels)


def paste_mask(small, box, threshold=0.5):
    '''Маска 28×28 розтягується під розмір рамки й вклеюється в полотно.'''
    x0, y0 = max(0, int(np.floor(box[0]))), max(0, int(np.floor(box[1])))
    x1, y1 = int(np.ceil(box[2])), int(np.ceil(box[3]))
    x1, y1 = min(SIZE, max(x0 + 1, x1)), min(SIZE, max(y0 + 1, y1))
    patch = torch.from_numpy(small.astype(np.float32))[None, None]
    back = F.interpolate(patch, size=(y1 - y0, x1 - x0), mode="bilinear",
                         align_corners=False)[0, 0].numpy()
    out = np.zeros((SIZE, SIZE), bool)
    out[y0:y1, x0:x1] = back >= threshold
    return out


def train_detector(data_pack, seed, epochs=DET_EPOCHS, batch_size=DET_BATCH,
                   learning_rate=3e-3, per_class=True):
    torch.manual_seed(seed)
    model = MaskDetector(per_class=per_class)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    images = data_pack["images"]
    started = time.time()
    model.train()
    for _epoch in range(epochs):
        order = torch.randperm(len(images))
        for start in range(0, len(images), batch_size):
            batch = order[start:start + batch_size]
            logits, distances, center, middle = model(images[batch])
            positive = data_pack["positive"][batch]
            loss = focal_loss(logits, data_pack["cls"][batch]).sum()
            if positive.any():
                loss = loss + F.smooth_l1_loss(distances[positive],
                                               data_pack["dist"][batch][positive],
                                               reduction="sum")
                loss = loss + F.binary_cross_entropy_with_logits(
                    center[positive], data_pack["center"][batch][positive],
                    reduction="sum")
            loss = loss / max(1, int(positive.sum().item()))

            # гілку маски вчимо на істинних рамках — так само, як у Mask R-CNN
            rois, targets, labels = [], [], []
            for slot, scene_index in enumerate(batch.tolist()):
                boxes = data_pack["gt_boxes"][scene_index]
                if len(boxes) == 0:
                    continue
                rois.append(torch.cat([torch.full((len(boxes), 1), float(slot)), boxes], dim=1))
                targets.append(data_pack["gt_masks"][scene_index])
                labels.append(data_pack["gt_labels"][scene_index])
            if rois:
                predicted = model.masks_for(middle, torch.cat(rois))
                labels = torch.cat(labels)
                if per_class:
                    predicted = predicted[torch.arange(len(labels)), labels]
                else:
                    predicted = predicted[:, 0]
                loss = loss + F.binary_cross_entropy_with_logits(predicted, torch.cat(targets))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return model, time.time() - started


@torch.no_grad()
def predict(model, data, score_floor=0.05, nms_threshold=0.5):
    '''Рамки з чотирьох відстаней, далі NMS, далі маска в кожній рамці.'''
    model.eval()
    images = torch.from_numpy(np.stack([scene["image"] for scene in data])[:, None])
    out = []
    for index in range(len(data)):
        logits, distances, center, middle = model(images[index:index + 1])
        probability = torch.sigmoid(logits[0])
        step = distances[0] * STRIDE
        boxes = torch.stack([POINTS[:, 0] - step[:, 0], POINTS[:, 1] - step[:, 1],
                             POINTS[:, 0] + step[:, 2], POINTS[:, 1] + step[:, 3]],
                            dim=1).clamp(0, SIZE)
        score, class_id = probability.max(dim=1)
        score = torch.sqrt(score * torch.sigmoid(center[0]))     # оцінка з centerness
        keep = (score >= score_floor) & (boxes[:, 2] - boxes[:, 0] > 1) \
            & (boxes[:, 3] - boxes[:, 1] > 1)
        boxes, score, class_id = boxes[keep], score[keep], class_id[keep]
        if len(boxes):
            keep = nms(boxes, score, nms_threshold)
            boxes, score, class_id = boxes[keep], score[keep], class_id[keep]
        masks = []
        if len(boxes):
            rois = torch.cat([torch.zeros(len(boxes), 1), boxes], dim=1)
            logit_maps = model.masks_for(middle, rois)
            for k in range(len(boxes)):
                channel = logit_maps[k, class_id[k]] if model.per_class else logit_maps[k, 0]
                masks.append(paste_mask(torch.sigmoid(channel).numpy(), boxes[k].numpy()))
        out.append(dict(boxes=boxes.numpy(), scores=score.numpy(),
                        labels=class_id.numpy(), masks=masks))
    return out


dense_pack = pack(dense_train)
print("цілі готові: %d сцен, %d позицій на сцену"
      % (len(dense_pack["images"]), POINTS.shape[0]))

### Навчання: два варіанти гілки маски, по три зерна

Перший варіант — канал на кожен клас, як у Mask R-CNN. Другий — один канал на
всіх. Питання просте: чи справді окремі канали щось дають.

In [ ]:
detector_results = {}
for per_class in (True, False):
    name = "канал на клас" if per_class else "один канал"
    box_scores, mask_scores, mask_range, seconds = [], [], [], []
    for seed in (0, 1, 2):
        model, elapsed = train_detector(dense_pack, seed, per_class=per_class)
        predictions = predict(model, dense_test)
        boxes_basket = collect_overlaps(predictions, dense_truth, "box")
        masks_basket = collect_overlaps(predictions, dense_truth, "mask")
        box_scores.append(ap_at(boxes_basket, 0.5))
        mask_scores.append(ap_at(masks_basket, 0.5))
        mask_range.append(ap_mean(masks_basket))
        seconds.append(elapsed)
        if per_class and seed == 0:
            top_down_model = model
            top_down_dense = predictions
        print("  %-14s зерно %d: %4.0f с   box AP %.4f   mask AP %.4f   mask AP[.50:.95] %.4f"
              % (name, seed, elapsed, box_scores[-1], mask_scores[-1], mask_range[-1]))
    detector_results[name] = dict(box=box_scores, mask=mask_scores, rng=mask_range)
    print("  %-14s середнє: box %.4f ±%.4f   mask %.4f ±%.4f   mask[.50:.95] %.4f"
          % (name, float(np.mean(box_scores)), max(box_scores) - min(box_scores),
             float(np.mean(mask_scores)), max(mask_scores) - min(mask_scores),
             float(np.mean(mask_range))))
    print()

gap = float(np.mean(detector_results["канал на клас"]["mask"])) \
    - float(np.mean(detector_results["один канал"]["mask"]))
spread = max(detector_results["канал на клас"]["mask"]) \
    - min(detector_results["канал на клас"]["mask"])
print("канал на клас проти одного каналу: різниця mask AP %.4f при розкиді %.4f"
      % (gap, spread))
print("Різниця, менша за розкид, різницею не є: на трьох простих класах")
print("розчіплювати клас і форму нема від чого.")

### Той самий детектор на рідких сценах

Перевірка себе. У [темі 26](../26-one-stage/lecture.html) та сама голова без гілки
маски дала на рідких сценах `mAP@0.5 = 0.9187`. Подивимось, скільки дає наша.

In [ ]:
sparse_top_down = predict(top_down_model, sparse_test)
sparse_box = collect_overlaps(sparse_top_down, sparse_truth, "box")
sparse_mask = collect_overlaps(sparse_top_down, sparse_truth, "mask")
print("на рідких сценах:  box AP@0.5 %.4f   mask AP@0.5 %.4f   mask AP[.50:.95] %.4f"
      % (ap_at(sparse_box, 0.5), ap_at(sparse_mask, 0.5), ap_mean(sparse_mask)))
dense_box_basket = collect_overlaps(top_down_dense, dense_truth, "box")
dense_mask_basket = collect_overlaps(top_down_dense, dense_truth, "mask")
print("на щільних сценах: box AP@0.5 %.4f   mask AP@0.5 %.4f   mask AP[.50:.95] %.4f"
      % (ap_at(dense_box_basket, 0.5), ap_at(dense_mask_basket, 0.5),
         ap_mean(dense_mask_basket)))
print()
print("Гілка маски рамкам не зашкодила. Складнішими є саме щільні сцени.")
print()

# чому одне зерно іноді провалює box AP, не провалюючи mask AP
predicted_area, truth_area = [], []
for prediction, scene in zip(top_down_dense, dense_test):
    confident = prediction["scores"] >= 0.5
    for box in prediction["boxes"][confident]:
        predicted_area.append((box[2] - box[0]) * (box[3] - box[1]))
    for box in scene["boxes"]:
        truth_area.append((box[2] - box[0]) * (box[3] - box[1]))
print("середня площа рамки: прогноз %.1f px², істина %.1f px²"
      % (float(np.mean(predicted_area)), float(np.mean(truth_area))))
print()
print("Навіть у найкращому зерні рамка виходить дрібнішою за істинну.")
print("Коли регресії не пощастило зовсім, рамка їде — і box AP валиться;")
print("маска при цьому фарбує всередині рамки лише сам предмет, тож mask AP тримається.")
print("Саме тому розкид box AP у варіанті «один канал» вийшов %.4f,"
      % (max(detector_results["один канал"]["box"]) - min(detector_results["один канал"]["box"])))
print("а розкид mask AP на тих самих трьох прогонах — лише %.4f."
      % (max(detector_results["один канал"]["mask"]) - min(detector_results["один канал"]["mask"])))

## 6 · Стеля роздільності маски — без жодного навчання

Mask R-CNN передбачає маску 28×28 незалежно від розміру предмета. Скільки на
цьому втрачається, можна порахувати наперед: беремо істинну маску, стискаємо до
R×R, розтягуємо назад і міряємо IoU з оригіналом.

In [ ]:
def shape_on_canvas(kind, side):
    '''Та сама фігура, що в сценах, намальована на полотні side×side.'''
    ys, xs = np.mgrid[0:side, 0:side]
    radius = (side - 1) / 2.0
    center = radius
    if kind == 0:
        return (xs - center) ** 2 + (ys - center) ** 2 <= radius * radius
    if kind == 1:
        return (np.abs(xs - center) <= radius) & (np.abs(ys - center) <= radius)
    return ((ys - center + radius >= 0) & (ys - center <= radius)
            & (np.abs(xs - center) <= (ys - center + radius) / 2.0))


def comb_shape(side, tooth=3):
    '''Гребінець: зубці сталої ширини 3 px на предметі будь-якого розміру.'''
    ys, xs = np.mgrid[0:side, 0:side]
    shape = np.ones((side, side), bool)
    shape[(xs // tooth) % 2 == 1] = False
    shape[ys < side // 4] = True                 # спинка гребінця
    return shape


def squeeze_and_back(mask, mask_side):
    '''Стискаємо маску до mask_side×mask_side і розтягуємо назад.'''
    patch = torch.from_numpy(mask.astype(np.float32))[None, None]
    small = F.interpolate(patch, size=(mask_side, mask_side), mode="bilinear",
                          align_corners=False)
    back = F.interpolate(small, size=mask.shape, mode="bilinear",
                         align_corners=False)[0, 0].numpy() >= 0.5
    return mask_iou(back, mask)


print("гладкі фігури (середнє по колу, квадрату й трикутнику):")
print("  предмет |  7×7   | 14×14  | 28×28  | 56×56")
for side in (16, 28, 56, 112, 224):
    row = [np.mean([squeeze_and_back(shape_on_canvas(k, side), out) for k in range(3)])
           for out in (7, 14, 28, 56)]
    print("  %3d px  | %.4f | %.4f | %.4f | %.4f" % (side, row[0], row[1], row[2], row[3]))
print()
print("Колонки не спадають: помилка живе на смузі вздовж межі, а ширина смуги")
print("росте разом із фігурою. Відносна втрата лишається сталою.")
print()
print("гребінець із зубцями сталої ширини 3 px:")
print("  предмет | зубців | 14×14  | 28×28  | 56×56")
for side in (28, 56, 112, 224):
    shape = comb_shape(side)
    print("  %3d px  |   %3d  | %.4f | %.4f | %.4f"
          % (side, side // 6, squeeze_and_back(shape, 14),
             squeeze_and_back(shape, 28), squeeze_and_back(shape, 56)))
print()
print("Ось де 28×28 справді ламається: щойно клітинка маски (сторона предмета / 28)")
print("стає ширшою за зубець, зубці зливаються в суцільну пластину.")

In [ ]:
sides = [box[2] - box[0] for scene in dense_test for box in scene["boxes"]]
print("наші предмети: сторона рамки %d…%d px, у середньому %.1f"
      % (min(sides), max(sides), float(np.mean(sides))))
for mask_side in (7, 14, 28, 56):
    values = []
    for scene in dense_test:
        for index, mask in enumerate(scene["masks"]):
            x0, y0, x1, y1 = [int(v) for v in scene["boxes"][index]]
            values.append(squeeze_and_back(mask[y0:y1, x0:x1], mask_side))
    print("  маска %2d×%-2d : IoU %.4f" % (mask_side, mask_side, float(np.mean(values))))
print()
print("У нашому зошиті 28×28 не коштує нічого: предмет менший за маску,")
print("тож стискання просто не відбувається. Стеля методу залежить від розміру")
print("предмета, і на дрібних предметах її немає.")

## 7 · RoI-Align проти RoI-пулінгу — саме на масках

[Тема 24](../24-two-stage/lecture.html) міряла зсув ознак. Для маски це
критичніше: маска — це і є геометрія.

Еталон пишемо руками: значення істинної маски в центрі кожної з 28×28 клітинок
рамки. З ним і порівнюємо те, що витягли обидва способи.

In [ ]:
def ideal_roi_mask(mask, box, out=28):
    '''Еталон: значення маски в центрі кожної з out×out клітинок рамки.'''
    x0, y0, x1, y1 = box
    step_x, step_y = (x1 - x0) / out, (y1 - y0) / out
    xs = x0 + (np.arange(out) + 0.5) * step_x
    ys = y0 + (np.arange(out) + 0.5) * step_y
    columns = np.clip(xs.astype(np.int64), 0, SIZE - 1)
    rows = np.clip(ys.astype(np.int64), 0, SIZE - 1)
    return mask[np.ix_(rows, columns)]


print("  зсув рамки | RoI-Align | RoI-пулінг | різниця")
for delta in (0.00, 0.25, 0.50, 0.75):
    align_values, pool_values = [], []
    for scene in dense_test:
        plane = torch.from_numpy(scene["seg"].astype(np.float32))[None, None]
        for index, mask in enumerate(scene["masks"]):
            box = np.clip(scene["boxes"][index].astype(np.float64) + delta, 0, SIZE)
            rois = torch.tensor([[0.0] + list(box)], dtype=torch.float32)
            class_plane = (plane == (scene["labels"][index] + 1)).float()
            aligned = roi_align(class_plane, rois, output_size=(28, 28), spatial_scale=1.0,
                                sampling_ratio=2, aligned=True)[0, 0].numpy() >= 0.5
            pooled = roi_pool(class_plane, rois, output_size=(28, 28),
                              spatial_scale=1.0)[0, 0].numpy() >= 0.5
            ideal = ideal_roi_mask(mask, box)
            align_values.append(mask_iou(aligned, ideal))
            pool_values.append(mask_iou(pooled, ideal))
    print("  %6.2f px  |  %.4f   |   %.4f   | %+.4f"
          % (delta, float(np.mean(align_values)), float(np.mean(pool_values)),
             float(np.mean(align_values)) - float(np.mean(pool_values))))
print()
print("Колонка RoI-Align рівна, колонка RoI-пулінгу стрибає: результат залежить")
print("не від моделі, а від того, чи пощастило рамці лягти близько до цілого числа.")
print()
print("Округлення в Python і в C різні, і саме на половинах:")
for value in (0.5, 1.5, 2.5, 3.5, 4.5):
    print("   round(%.1f) = %d   а math.floor(%.1f + 0.5) = %d"
          % (value, round(value), value, math.floor(value + 0.5)))

## 8 · Скільки коштує маска в справжньому Mask R-CNN

Обидві моделі збираємо офлайн, без ваг: `weights=None, weights_backbone=None`.
Різниця між ними і є ціною гілки маски.

In [ ]:
from torchvision.models.detection import maskrcnn_resnet50_fpn, fasterrcnn_resnet50_fpn


def count_parameters(module):
    return sum(p.numel() for p in module.parameters())


mask_rcnn = maskrcnn_resnet50_fpn(weights=None, weights_backbone=None)
faster_rcnn = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None)

box_head_total = (count_parameters(mask_rcnn.roi_heads.box_head)
                  + count_parameters(mask_rcnn.roi_heads.box_predictor))
mask_head_total = (count_parameters(mask_rcnn.roi_heads.mask_head)
                   + count_parameters(mask_rcnn.roi_heads.mask_predictor))
total = count_parameters(mask_rcnn)

print("maskrcnn_resnet50_fpn   %10d" % total)
print("fasterrcnn_resnet50_fpn %10d" % count_parameters(faster_rcnn))
print("різниця                 %10d" % (total - count_parameters(faster_rcnn)))
print("голова масок            %10d   ← те саме число" % mask_head_total)
print("   з них mask_head      %10d" % count_parameters(mask_rcnn.roi_heads.mask_head))
print("   з них mask_predictor %10d" % count_parameters(mask_rcnn.roi_heads.mask_predictor))
assert total - count_parameters(faster_rcnn) == mask_head_total, "різниця не сходиться!"
print("✅ Mask R-CNN — це буквально Faster R-CNN плюс одна гілка")
print()
parts = [("тіло ResNet-50", count_parameters(mask_rcnn.backbone.body)),
         ("FPN", count_parameters(mask_rcnn.backbone.fpn)),
         ("RPN", count_parameters(mask_rcnn.rpn)),
         ("голова рамок", box_head_total),
         ("голова масок", mask_head_total)]
for name, value in parts:
    print("  %-16s %10d   %5.1f %%" % (name, value, 100 * value / total))
print("  %-16s %10d   100.0 %%" % ("усього", total))
print()
print("розмір RoI для рамок: %s, для маски: %s"
      % (mask_rcnn.roi_heads.box_roi_pool.output_size,
         mask_rcnn.roi_heads.mask_roi_pool.output_size))
print("останній шар голови масок:", mask_rcnn.roi_heads.mask_predictor.mask_fcn_logits)
print()
classes = mask_rcnn.roi_heads.mask_predictor.mask_fcn_logits.out_channels
per_class_mask = 256 + 1                       # 256 множень і один зсув на канал
per_class_box = 1025 * 5                       # клас плюс чотири координати
print("класів у моделі: %d" % classes)
print("клас коштує масці %d ваг, а рамці %d — у %.0f разів більше"
      % (per_class_mask, per_class_box, per_class_box / per_class_mask))
print("усі %d каналів маски разом: %d ваг, тобто %.2f %% моделі"
      % (classes, per_class_mask * classes, 100 * per_class_mask * classes / total))
print()
print("якби клас був один, голова масок схудла б на %d ваг, а голова рамок — на %d"
      % (per_class_mask * (classes - 1), per_class_box * (classes - 1)))

## 9 · Знизу вгору: пікселі голосують за центр

Інший бік думки: не шукати предмет цілком, а попросити кожен піксель показати на
центр свого предмета. Пікселі одного предмета покажуть приблизно в одну точку,
пікселі різних — у різні. Скільки згущень голосів — стільки предметів.

Мережа — маленький U-Net на два рівні стискання зі скіпами
([тема 30](../30-unet/lecture.html)) із двома виходами: клас пікселя й два числа
зсуву до центра.

In [ ]:
NUM_LABELS = CLASS_COUNT + 1                   # три предмети плюс фон
ROWS, COLUMNS = np.mgrid[0:SIZE, 0:SIZE]


class VoteNet(nn.Module):
    '''Клас кожного пікселя і зсув від пікселя до центра його предмета.'''

    def __init__(self, width=16):
        super().__init__()

        def block(in_channels, out_channels):
            return nn.Sequential(nn.Conv2d(in_channels, out_channels, 3, padding=1),
                                 nn.BatchNorm2d(out_channels), nn.ReLU(),
                                 nn.Conv2d(out_channels, out_channels, 3, padding=1),
                                 nn.BatchNorm2d(out_channels), nn.ReLU())

        self.down1 = block(1, width)
        self.down2 = block(width, width * 2)
        self.bottom = block(width * 2, width * 4)
        self.up2 = block(width * 4 + width * 2, width * 2)
        self.up1 = block(width * 2 + width, width)
        self.semantic = nn.Conv2d(width, NUM_LABELS, 1)
        self.offset = nn.Conv2d(width, 2, 1)

    def forward(self, x):
        first = self.down1(x)                                   # 64×64
        second = self.down2(F.max_pool2d(first, 2))             # 32×32
        deep = self.bottom(F.max_pool2d(second, 2))             # 16×16
        deep = F.interpolate(deep, scale_factor=2, mode="nearest")
        second = self.up2(torch.cat([deep, second], dim=1))     # скіп із 32×32
        second = F.interpolate(second, scale_factor=2, mode="nearest")
        first = self.up1(torch.cat([second, first], dim=1))     # скіп із 64×64
        return self.semantic(first), self.offset(first)


def pack_votes(data):
    '''Ціль зсуву: від кожного пікселя предмета до центра мас цього предмета.'''
    images = torch.from_numpy(np.stack([scene["image"] for scene in data])[:, None])
    segmentation = torch.from_numpy(np.stack([scene["seg"] for scene in data]))
    offsets = np.zeros((len(data), 2, SIZE, SIZE), np.float32)
    for index, scene in enumerate(data):
        for mask in scene["masks"]:
            rows, columns = np.nonzero(mask)
            offsets[index, 0][mask] = columns.mean() - COLUMNS[mask]
            offsets[index, 1][mask] = rows.mean() - ROWS[mask]
    return images, segmentation, torch.from_numpy(offsets)


def train_votes(images, segmentation, offsets, seed, epochs=VOTE_EPOCHS,
                batch_size=VOTE_BATCH, learning_rate=3e-3):
    torch.manual_seed(seed)
    model = VoteNet()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    started = time.time()
    model.train()
    for _epoch in range(epochs):
        order = torch.randperm(len(images))
        for start in range(0, len(images), batch_size):
            batch = order[start:start + batch_size]
            logits, predicted_offsets = model(images[batch])
            loss = F.cross_entropy(logits, segmentation[batch])
            foreground = segmentation[batch] > 0
            if foreground.any():
                weight = foreground.unsqueeze(1).float()
                # зсуви вчимо тільки на пікселях предметів: у фону центра немає
                loss = loss + (F.l1_loss(predicted_offsets, offsets[batch],
                                         reduction="none") * weight).sum() \
                    / weight.sum().clamp(min=1) / 8.0
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return model, time.time() - started


print("параметрів у VoteNet: %d" % sum(p.numel() for p in VoteNet().parameters()))

### Групування: від голосів до предметів

Зводимо голоси в сітку 64×64, злегка розмиваємо й беремо локальні максимуми — це
центри. Далі кожен піксель приписуємо до найближчого до його голосу центра.
Кількість предметів ніхто не задавав: вона випала з даних.

In [ ]:
def find_peaks(votes_x, votes_y, min_votes=25):
    '''Локальні максимуми в сітці голосів — це й будуть центри предметів.'''
    accumulator = np.zeros((SIZE, SIZE), np.float32)
    columns = np.clip(np.round(votes_x).astype(int), 0, SIZE - 1)
    rows = np.clip(np.round(votes_y).astype(int), 0, SIZE - 1)
    np.add.at(accumulator, (rows, columns), 1.0)
    smooth = F.avg_pool2d(torch.from_numpy(accumulator)[None, None], 3, 1, 1)
    highest = F.max_pool2d(smooth, 5, 1, 2)
    is_peak = ((smooth == highest) & (smooth * 9 >= min_votes))[0, 0].numpy()
    return list(zip(*np.nonzero(is_peak)))


@torch.no_grad()
def group_scene(model, image, min_votes=25):
    '''Один прохід мережі, потім групування голосів у предмети.'''
    model.eval()
    logits, offsets = model(torch.from_numpy(image)[None, None])
    labels = logits[0].argmax(dim=0).numpy()
    offsets = offsets[0].numpy()
    masks, out_labels, boxes, scores = [], [], [], []
    for class_index in range(1, NUM_LABELS):
        selected = labels == class_index
        if selected.sum() < 8:
            continue
        votes_x = COLUMNS[selected] + offsets[0][selected]
        votes_y = ROWS[selected] + offsets[1][selected]
        peaks = find_peaks(votes_x, votes_y, min_votes)
        if not peaks:
            peaks = [(int(round(votes_y.mean())), int(round(votes_x.mean())))]
        peak_y = np.array([p[0] for p in peaks], np.float32)
        peak_x = np.array([p[1] for p in peaks], np.float32)
        distance = (votes_x[:, None] - peak_x[None, :]) ** 2 \
            + (votes_y[:, None] - peak_y[None, :]) ** 2
        owner = distance.argmin(axis=1)
        rows, columns = np.nonzero(selected)
        for peak_index in range(len(peaks)):
            chosen = owner == peak_index
            if chosen.sum() < 8:            # надто дрібні згустки — це шум
                continue
            mask = np.zeros((SIZE, SIZE), bool)
            mask[rows[chosen], columns[chosen]] = True
            masks.append(mask)
            out_labels.append(class_index - 1)
            boxes.append([float(columns[chosen].min()), float(rows[chosen].min()),
                          float(columns[chosen].max() + 1), float(rows[chosen].max() + 1)])
            scores.append(float(chosen.sum()))
    return dict(masks=masks, labels=np.array(out_labels, np.int64),
                boxes=np.array(boxes, np.float32).reshape(-1, 4),
                scores=np.array(scores, np.float32))


vote_images, vote_seg, vote_offsets = pack_votes(dense_train)
dense_bottom_ap, dense_bottom_rng, sparse_bottom_ap, sparse_bottom_rng = [], [], [], []
for seed in (0, 1, 2):
    model, elapsed = train_votes(vote_images, vote_seg, vote_offsets, seed)
    dense_predictions = [group_scene(model, scene["image"]) for scene in dense_test]
    sparse_predictions = [group_scene(model, scene["image"]) for scene in sparse_test]
    dense_basket = collect_overlaps(dense_predictions, dense_truth, "mask")
    sparse_basket = collect_overlaps(sparse_predictions, sparse_truth, "mask")
    dense_bottom_ap.append(ap_at(dense_basket, 0.5))
    dense_bottom_rng.append(ap_mean(dense_basket))
    sparse_bottom_ap.append(ap_at(sparse_basket, 0.5))
    sparse_bottom_rng.append(ap_mean(sparse_basket))
    if seed == 0:
        bottom_dense = dense_predictions
    print("  зерно %d: %4.0f с   щільні mask AP %.4f (AP[.50:.95] %.4f)   рідкі %.4f"
          % (seed, elapsed, dense_bottom_ap[-1], dense_bottom_rng[-1], sparse_bottom_ap[-1]))
print("  середнє: щільні %.4f ±%.4f, [.50:.95] %.4f;  рідкі %.4f ±%.4f, [.50:.95] %.4f"
      % (float(np.mean(dense_bottom_ap)), max(dense_bottom_ap) - min(dense_bottom_ap),
         float(np.mean(dense_bottom_rng)), float(np.mean(sparse_bottom_ap)),
         max(sparse_bottom_ap) - min(sparse_bottom_ap), float(np.mean(sparse_bottom_rng))))

## 10 · Три способи поруч

Останній замір теми: усі три способи на тих самих сценах, обидві щільності,
обидва пороги.

In [ ]:
sparse_blob_basket = collect_overlaps(sparse_blobs, sparse_truth, "mask")
dense_blob_basket = collect_overlaps(dense_blobs, dense_truth, "mask")

print("                     щільні сцени              рідкі сцени")
print("  спосіб            AP@0.5   AP[.50:.95]     AP@0.5   AP[.50:.95]")
print("  " + "-" * 66)
print("  звʼязні плями     %.4f      %.4f        %.4f      %.4f"
      % (ap_at(dense_blob_basket, 0.5), ap_mean(dense_blob_basket),
         ap_at(sparse_blob_basket, 0.5), ap_mean(sparse_blob_basket)))
print("  зверху вниз       %.4f      %.4f        %.4f      %.4f"
      % (float(np.mean(detector_results["канал на клас"]["mask"])),
         float(np.mean(detector_results["канал на клас"]["rng"])),
         ap_at(sparse_mask, 0.5), ap_mean(sparse_mask)))
print("  знизу вгору       %.4f      %.4f        %.4f      %.4f"
      % (float(np.mean(dense_bottom_ap)), float(np.mean(dense_bottom_rng)),
         float(np.mean(sparse_bottom_ap)), float(np.mean(sparse_bottom_rng))))
print()
print("При порозі 0.5 зверху вниз і звʼязні плями майже рівні.")
print("При суворому усередненні плями обходять зверху вниз: там, де вони не")
print("помилились, маска в них точна до пікселя, а зверху вниз щоразу")
print("перемальовує маску з клаптика 28×28 усередині рамки.")
print()
print("час зошита наскрізь: %.0f с" % (time.time() - NOTEBOOK_STARTED))

### Як це виглядає

Одна щільна сцена, розібрана трьома способами. Кольори — окремі предмети.

In [ ]:
def to_map(prediction, score_floor=0.0):
    '''Кожен предмет своїм номером. Для детектора беремо лише впевнені прогнози —
    інакше карту закриють десятки слабких дублікатів, які метриці не заважають,
    бо стоять у кінці списку, а картинку роблять нечитабельною.'''
    out = np.zeros((SIZE, SIZE), np.int64)
    number = 0
    order = np.argsort(-prediction["scores"])
    for index in order:
        if prediction["scores"][index] < score_floor:
            continue
        number += 1
        out[prediction["masks"][index]] = number
    return out


scene_index = 4
figure, axes = plt.subplots(1, 5, figsize=(15, 3.2))
scene = dense_test[scene_index]
axes[0].imshow(scene["image"], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("сцена", fontsize=9)
axes[1].imshow(to_map(dense_truth[scene_index]), cmap="tab10", vmin=0, vmax=9)
axes[1].set_title("істина: %d" % len(scene["masks"]), fontsize=9)
axes[2].imshow(to_map(dense_blobs[scene_index]), cmap="tab10", vmin=0, vmax=9)
axes[2].set_title("плями: %d" % len(dense_blobs[scene_index]["masks"]), fontsize=9)
confident = int((top_down_dense[scene_index]["scores"] >= 0.5).sum())
axes[3].imshow(to_map(top_down_dense[scene_index], 0.5), cmap="tab10", vmin=0, vmax=9)
axes[3].set_title("зверху вниз: %d" % confident, fontsize=9)
axes[4].imshow(to_map(bottom_dense[scene_index]), cmap="tab10", vmin=0, vmax=9)
axes[4].set_title("знизу вгору: %d" % len(bottom_dense[scene_index]["masks"]), fontsize=9)
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()
print("на цій сцені предметів %d, плям %d, знайдено знизу вгору %d"
      % (len(scene["masks"]), len(dense_blobs[scene_index]["masks"]),
         len(bottom_dense[scene_index]["masks"])))
print("зверху вниз видав %d рамок з оцінкою не менше 0.5"
      % int((top_down_dense[scene_index]["scores"] >= 0.5).sum()))

## Завдання

**🟢 Рівень 1.** Прожени `max_objects` = 2, 3, 4, 5, 6, 8 і побудуй криву: частка
предметів, що торкаються предмета свого класу, залежно від щільності. Де ця
частка переходить за 20 %?

**🟡 Рівень 2.** Пороги в `group_scene` (`min_votes` і мінімальний розмір згустка)
взяті зі стелі. Перебери `min_votes` зі значень 8, 15, 25, 40, 60 і знайди, де
mask AP найвища. Поясни, що ламається на кожному краю.

**🔴 Рівень 3.** Напиши `mask IoU` і `mask AP` з нуля, звір їх на прикладі,
порахованому руками, — і порівняй зверху вниз проти знизу вгору на **своїх**
сценах, у яких предмети не опуклі: додай у `shape_mask` кільце й хрест. Чи
збережеться перевага групування? Три зерна на кожен спосіб.

Повний опис — у [homework.html](homework.html).